## Metode inference ini menggunakan pendekatan SHAP explainer sebagai fungsi utama reasoning.
## Codingan menggunakan metode dari "shap_explainer.joblib" daripada menggunakan "explanation_baselines.json".

In [1]:
import json
import joblib
import numpy as np
import pandas as pd
import shap
from pathlib import Path
from IPython.display import display

# Konfigurasi + Load Artifacts
def print_section(title):
    print("=" * 50)
    print(title)
    print("=" * 50)

# Tentukan path folder artifacts
artifact_dir = Path("../artifacts/post_award_anomaly")

print_section("LOADING ARTIFACTS")

# Load Model, Preprocessor, dan SHAP Explainer
model = joblib.load(artifact_dir / "isolation_forest.joblib")
preprocessor = joblib.load(artifact_dir / "preprocessor.joblib")
explainer_shap = joblib.load(artifact_dir / "shap_explainer.joblib")

# Load Configuration & Metadata
model_config = json.loads((artifact_dir / "model_config.json").read_text(encoding="utf-8"))
explanation_meta = json.loads((artifact_dir / "explanation_meta.json").read_text(encoding="utf-8"))

# Load data demo untuk testing
demo_input = pd.read_csv("../data/cleaned/test/test_data_127.csv")

print(f"Artifacts dir        : {artifact_dir.resolve()}")
print(f"Anomaly threshold    : {model_config['anomaly_threshold']:.6f}")
print(f"Feature Names        : {len(explanation_meta['feature_names_preprocessed'])} features loaded.")

LOADING ARTIFACTS


FileNotFoundError: [Errno 2] No such file or directory: '..\\artifacts\\post_award_anomaly\\isolation_forest.joblib'

In [ ]:
# Helper Function
def format_number(value):
    if pd.isna(value): return "missing"
    if isinstance(value, (int, np.integer)): return f"{int(value):,}"
    if isinstance(value, (float, np.floating)): return f"{value:,.2f}"
    return str(value)

def assign_severity(scores, medium_cutoff, anomaly_threshold):
    return np.select(
        [scores >= anomaly_threshold, scores >= medium_cutoff],
        ["high", "medium"],
        default="low"
    )

def engineer_features(frame):
    data = frame.copy()
    data["award_date"] = pd.to_datetime(data["award_date"])
    data["award_month"] = data["award_date"].dt.month
    data["award_quarter"] = data["award_date"].dt.quarter
    data["award_weekday"] = data["award_date"].dt.weekday
    data["log_tender_minvalue"] = np.log1p(data["tender_minvalue"])
    data["log_award_value"] = np.log1p(data["award_value"])
    data["value_gap"] = data["award_value"] - data["tender_minvalue"]
    data["title_word_count"] = data["tender_title"].fillna("").str.split().str.len()
    data["award_title_word_count"] = data["award_title"].fillna("").str.split().str.len()
    data["supplier_count"] = data["award_supplier"].fillna("").astype(str).str.split(",").str.len()
    data["award_value_per_day"] = data["award_value"] / data["days_to_award"].replace(0, 1)
    data["same_day_award_flag"] = (data["days_to_award"] == 0).astype(int)
    return data

In [ ]:
# SHAP Explainer
# 1. Mapping Nama Fitur ke Bahasa Manusia (Business/Audit Terms)
KAMUS_KONSEP = {
    "award_title_word_count": "Kompleksitas Judul Kontrak",
    "days_to_award": "Durasi Proses Tender",
    "budget_utilization_ratio": "Rasio Penyerapan Anggaran",
    "value_gap": "Selisih Nilai Tender dan Kontrak",
    "supplier_count": "Jumlah Peserta Tender",
    "award_value_per_day": "Laju Pengeluaran Harian",
    "tender_minvalue": "Batas Minimum Tender",
    "award_value": "Nilai Kontrak Akhir"
}

# 2. Mapping Risiko Berdasarkan Analisis Audit
KAMUS_RISIKO = {
    "award_title_word_count": "pola penamaan judul yang tidak standar seringkali digunakan untuk mengaburkan spesifikasi asli proyek.",
    "days_to_award": "proses yang selesai terlalu cepat atau terlalu lambat mengindikasikan adanya potensi pengaturan pemenang (bid-rigging).",
    "budget_utilization_ratio": "penyerapan yang mendekati 100% secara sempurna merupakan indikator umum terjadinya mark-up harga.",
    "value_gap": "selisih yang tidak proporsional menunjukkan potensi inefisiensi atau kesalahan estimasi biaya awal.",
    "supplier_count": "minimnya partisipan tender dapat mengurangi kompetisi sehat dan meningkatkan risiko monopoli.",
    "award_value_per_day": "beban biaya harian yang ekstrem menunjukkan ketidakwajaran antara nilai proyek dengan durasi pengerjaan.",
    "tender_minvalue": "penetapan batas minimum yang tidak lazim berisiko membatasi partisipasi vendor kompeten."
}

def generate_natural_reason(feat, raw_val, shap_val, severity_band):
    """Menerjemahkan kontribusi SHAP menjadi narasi chatbot berdasarkan 3 tingkat risiko."""
    is_anomaly_driver = shap_val > 0
    nama_manusiawi = KAMUS_KONSEP.get(feat, feat.replace("_", " ").title())
    val_str = format_number(raw_val)
    
    # KASUS 1: RISIKO TINGGI (ANOMALY)
    if severity_band == "high":
        if is_anomaly_driver:
            risiko = KAMUS_RISIKO.get(feat, "hal ini memerlukan verifikasi kepatuhan dokumen lebih lanjut.")
            return (f"Sistem menemukan indikasi penyimpangan serius pada **{nama_manusiawi}** ({val_str}). "
                    f"Secara audit, {risiko}")
        else:
            return f"Meskipun terdapat temuan risiko lain, parameter **{nama_manusiawi}** ({val_str}) terpantau tetap stabil."

    # KASUS 2: RISIKO SEDANG (MEDIUM / BORDERLINE)
    elif severity_band == "medium":
        if is_anomaly_driver:
            return (f"Terdapat sedikit kejanggalan pada data **{nama_manusiawi}** ({val_str}). "
                    f"Meskipun menunjukkan pola yang tidak biasa, angka ini dinilai masih berada dalam batas toleransi kebijakan.")
        else:
            return f"Parameter **{nama_manusiawi}** ({val_str}) memberikan sinyal kestabilan di tengah beberapa anomali minor lainnya."

    # KASUS 3: RISIKO RENDAH (LOW / NORMAL)
    else:
        if is_anomaly_driver:
            return f"Komponen **{nama_manusiawi}** ({val_str}) menunjukkan aktivitas yang dinamis namun tetap sesuai dengan standar operasional."
        else:
            return f"Parameter **{nama_manusiawi}** ({val_str}) sangat identik dengan profil pengadaan yang bersih dan akuntabel."

def explain_prediction_shap(row_idx, processed_row, original_row, shap_values):
    """Mengambil 3 alasan utama berdasarkan SHAP values dengan membedakan 3 tingkatan bahasa."""
    row_shap = shap_values[row_idx]
    feature_names = explanation_meta["feature_names_preprocessed"]
    
    # Ambil Severity Band (low, medium, high)
    severity_band = original_row['severity_band']
    
    # Ambil index dengan nilai SHAP tertinggi (pendorong keputusan paling kuat)
    top_indices = np.argsort(row_shap)[-3:][::-1]
    
    reasons = []
    for idx in top_indices:
        feat_name = feature_names[idx]
        raw_val = original_row[feat_name] if feat_name in original_row else "N/A"
        # Menambahkan parameter severity_band untuk menentukan gaya bahasa
        reasons.append(generate_natural_reason(feat_name, raw_val, row_shap[idx], severity_band))
    
    # Header narasi berdasarkan tingkat risiko
    if severity_band == "high":
        header = "Sistem mendeteksi aktivitas yang MENCURIGAKAN dan berisiko tinggi:"
    elif severity_band == "medium":
        header = "Sistem menemukan beberapa temuan BORDERLINE yang memerlukan perhatian moderat:"
    else:
        header = "Status transaksi dinilai AMAN dan memenuhi kriteria kepatuhan standar:"

    summary = f"[{severity_band.upper()}] {header}\n" + "\n".join([f"• {r}" for r in reasons])
    return summary

In [ ]:
# Mapping Hasil
print_section("RUNNING INFERENCE")

# 1. Engineering Fitur
demo_features = engineer_features(demo_input)
feature_columns = model_config["numeric_features"] + model_config["categorical_features"]

# 2. Transformasi & Prediksi
X_demo = preprocessor.transform(demo_features[feature_columns])
demo_scores = -model.score_samples(X_demo)

# 3. Hitung SHAP Values untuk Penjelasan
shap_values = explainer_shap.shap_values(X_demo)

# 4. Mapping Hasil
demo_features["anomaly_score"] = demo_scores
demo_features["prediction_label"] = np.where(demo_scores >= model_config["anomaly_threshold"], "anomaly", "normal")
demo_features["severity_band"] = assign_severity(demo_scores, model_config["medium_cutoff"], model_config["anomaly_threshold"])

# ---> TAMBAHKAN KODE INI UNTUK MENGATASI ERROR <---
# Membuat label kasus secara otomatis (contoh: Kasus 1, Kasus 2, dst)
demo_features["demo_case_label"] = [f"Kasus {i+1}" for i in range(len(demo_features))]

# 5. Generate Penjelasan untuk setiap baris
explanations = []
for i in range(len(demo_features)):
    reason_text = explain_prediction_shap(i, X_demo[i], demo_features.iloc[i], shap_values)
    explanations.append(reason_text)

demo_features["human_readable_explanation"] = explanations

# Tampilkan Hasil Akhir
display_cols = [
    "demo_case_label", "award_date", "mainprocurementcategory", 
    "award_value", "anomaly_score", "prediction_label", "human_readable_explanation"
]
display(demo_features[display_cols])

# Simpan hasil
demo_features.to_csv(artifact_dir / "inference_output_final.csv", index=False)
print(f"\nHasil inference disimpan ke: {artifact_dir / 'inference_output_final.csv'}")

In [ ]:
print_section("DETAILED CHATBOT EXPLANATIONS")
for idx, row in demo_features.iterrows():
    print(f"Kasus: {row['demo_case_label']}")
    print(f"Status: {row['prediction_label'].upper()} (Score: {row['anomaly_score']:.4f})")
    print(f"Tingkat Risiko: {row['severity_band'].upper()}")
    print(f"Analisis Sistem:\n{row['human_readable_explanation']}")
    print("-" * 50)